In [1]:
import pandas as pd
import os
from time import time

In [2]:
# Get the absolute path of the script
current_dir = os.getcwd()

# Move up two levels to the project root
project_root = os.path.abspath(os.path.join(current_dir, "../.."))

# Construct the CSV path
csv_path = os.path.join(project_root, "data", "reviews_labeled_cleaned.csv")

In [3]:
df = pd.read_csv(csv_path)
df.head()

,source,size,comment,original,Satisfaction,Authenticity,Type,needs_response,Category,Impact,Language,Emotion,cleaned
0,Friedom,short,Not salty enough,Not salty enough,3,Legitimate,Complaint,Yes,Product Quality / Food Quality,Moderate,English,Disappointment,Not salty enough
1,Friedom,short,Overall good chicken and fries,Overall good chicken and fries,4,Legitimate,Praise,No,Product Quality / Food Quality,No issue,English,Satisfaction,Overall good chicken and fries
2,Friedom,medium,Over rated\nTiles that are sticky to each othe...,Over rated\nطوابل لازقين فبعضاهم و مزيتتين ميش...,1,Legitimate,Complaint,Yes,"Customer Service, Product Quality / Food Quali...",High,"English, Derja",Frustration,Over rated طوابل لازقين فبعضاهم و مزيتتين ميش ...
3,Friedom,short,Food took +30 minutes to come.,Food took +30 minutes to come.,2,Legitimate,Complaint,Yes,Delivery / Logistics,Low Impact,English,Disappointment,Food took 30 minutes to come.
4,Friedom,long,"●●●●Food : ● the buns are so soggy ""DEAD"" like...","●●●●Food : ● the buns are so soggy ""DEAD"" like...",2,Legitimate,Complaint,Yes,"Customer Service, Product Quality / Food Quali...",High,English,Anger,Food the buns are so soggy DEAD like they cant...


In [4]:
df.shape

(1005, 13)

In [6]:
# Detect language of the reviews
from langdetect import detect
from textblob import TextBlob
from fasttext import load_model


def detect_lang(text):
    try:
        return detect(text)
    except:
        return "unknown"
    

def detect_fasttext(text):
    model = load_model('lid.176.bin')
    lang = model.predict(text)[0][0]
    return lang

In [7]:
durations = {}

In [8]:
time_start = time()
df['lang'] = df['cleaned'].apply(detect_lang)
time_end = time()
durations['langdetect'] = time_end - time_start

In [9]:
from fast_langdetect import detect, detect_multilingual, LangDetector, LangDetectConfig, DetectError


def detect_fastlangdetect(text):
    config = LangDetectConfig()
    detector = LangDetector(config)
    
    try:
        lang = detector.detect(text)
        return lang.get('lang')
    except DetectError:
        return "unknown"

/home/khaledbouabdallah/Projects/reviews_analysis_platform/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
start_time = time()
df['lang_fasttext'] = df['cleaned'].apply(detect_fasttext)
end_time = time()
durations['fasttext'] = end_time - start_time
df['lang_fasttext'] = df['lang_fasttext'].apply(lambda x: x.replace("__label__", ""))

ValueError: lid.176.bin cannot be opened for loading!

In [11]:
start_time = time()
df['fastlangdetect'] = df['cleaned'].apply(detect_fastlangdetect)
end_time = time()
durations['cld3'] = end_time - start_time

fast-langdetect: Text may be too long. Consider passing only a single sentence for accurate prediction.
fast-langdetect: Text may be too long. Consider passing only a single sentence for accurate prediction.
fast-langdetect: Text may be too long. Consider passing only a single sentence for accurate prediction.
fast-langdetect: Text may be too long. Consider passing only a single sentence for accurate prediction.
fast-langdetect: Text may be too long. Consider passing only a single sentence for accurate prediction.
fast-langdetect: Text may be too long. Consider passing only a single sentence for accurate prediction.
fast-langdetect: Text may be too long. Consider passing only a single sentence for accurate prediction.
fast-langdetect: Text may be too long. Consider passing only a single sentence for accurate prediction.
fast-langdetect: Text may be too long. Consider passing only a single sentence for accurate prediction.
fast-langdetect: Text may be too long. Consider passing only a s

In [12]:
df.to_csv(os.path.join(project_root, "data", "reviews_labeled_cleaned_bench_language.csv"), index=False)

## Postprocessing 


In [13]:
# Load the data back to skip first part
df = pd.read_csv(os.path.join(project_root, "data", "reviews_labeled_cleaned_bench_language.csv"))

In [14]:
lang_name_to_code = {
    "English": "en",
    "French": "fr",
    "Spanish": "es",
    "German": "de",
    "Italian": "it",
    "Dutch": "nl",
    "Portuguese": "pt",
    "Russian": "ru",
    "Japanese": "ja",
    "Chinese": "zh",
    "Korean": "ko",
    "Arabic": "ar",
    "Turkish": "tr",
    "Derja": "dz",
}

def lang_name_to_code_func(lang_name):
    languages = lang_name.split(',')
    languages = [lang.strip() for lang in languages]
    return ','.join([lang_name_to_code[lang_name] for lang_name in languages])


def land_code_standardize(code):
    if code == 'zh-cn':
        return 'zh'
    elif '-latn' in code:
        return code.replace('-latn','')
    else:
        return code
    

In [15]:
language_columns = ['lang', 'lang_langid', 'lang_polyglot', 'lang_cld3', 'lang_fasttext','fastlangdetect']

In [16]:
df["Language"] = df["Language"].apply(lang_name_to_code_func)

for column in language_columns:
    if column in df.columns:
        df[column] = df[column].apply(land_code_standardize)
    

In [17]:
df.head()

,source,size,comment,original,Satisfaction,Authenticity,Type,needs_response,Category,Impact,Language,Emotion,cleaned,lang,fastlangdetect
0,Friedom,short,Not salty enough,Not salty enough,3,Legitimate,Complaint,Yes,Product Quality / Food Quality,Moderate,en,Disappointment,Not salty enough,en,en
1,Friedom,short,Overall good chicken and fries,Overall good chicken and fries,4,Legitimate,Praise,No,Product Quality / Food Quality,No issue,en,Satisfaction,Overall good chicken and fries,en,en
2,Friedom,medium,Over rated\nTiles that are sticky to each othe...,Over rated\nطوابل لازقين فبعضاهم و مزيتتين ميش...,1,Legitimate,Complaint,Yes,"Customer Service, Product Quality / Food Quali...",High,"en,dz",Frustration,Over rated طوابل لازقين فبعضاهم و مزيتتين ميش ...,ar,ar
3,Friedom,short,Food took +30 minutes to come.,Food took +30 minutes to come.,2,Legitimate,Complaint,Yes,Delivery / Logistics,Low Impact,en,Disappointment,Food took 30 minutes to come.,en,en
4,Friedom,long,"●●●●Food : ● the buns are so soggy ""DEAD"" like...","●●●●Food : ● the buns are so soggy ""DEAD"" like...",2,Legitimate,Complaint,Yes,"Customer Service, Product Quality / Food Quali...",High,en,Anger,Food the buns are so soggy DEAD like they cant...,en,en


# Score calculations and results

In [19]:
def score(predicted_language, ground_truth_languages):
    ''' Language detection score calculation
    1: if the two columns match
    0.5: if the predicted_language is in the list of ground truth
    0: otherwise
    '''
    truth_language_list = ground_truth_languages.split(',')
    
    if predicted_language in truth_language_list:
        return 1 if len(truth_language_list) == 1 else 0.5
    else:
        return 0


In [20]:
for column in language_columns:
    if column in df.columns:
        df[column + "_score"] = df.apply(lambda x: score(x[column], x["Language"]), axis=1)
    

In [21]:
for column in language_columns:
    if column in df.columns:
        print(column + "_score", df[column + "_score"].mean())

lang_score 0.854726368159204
fastlangdetect_score 0.890049751243781


## calculations time

- langdetect = 2.9s
- langid = 20min
- ployglot = 0.16s
- cld3 score = 0.1s
- fasttext = 54s
- fast-langdetect = 9s

## notes

- fasttext model _lid176.bin_"_ is the best model with a _0.89_ average score with an average 0.05sec response time 

## LLM scroes:

- qwen2.5:3b-instruct: got 0.844 score in 47sec. most mistakes where not predicting dz at all (just arabic). over predicting arabic for english and french comments
- gemma3:27b: got 0.92 in 16min. can be improved with better prompt & postprocessing.

# Benchmark results analysis in depth

check where the models make wrong predictions

In [28]:
df.groupby("Language").agg({"Language":"count","lang_score": "sum", "lang_langid_score": "sum", "lang_polyglot_score": "sum", "lang_cld3_score": "sum", "lang_fasttext_score": "sum"})

KeyError: "Column(s) ['lang_cld3_score', 'lang_fasttext_score', 'lang_langid_score', 'lang_polyglot_score'] do not exist"

## notes

- as expected models do not have a dz (algerian derja) or any other code for algerian derja (fasttext model have egyptian arabic and standard arabic)

# Error Analysis

In [1]:
df_test = df[df['Language'] == 'en'][['cleaned', 'Language', 'lang', 'lang_langid', 'lang_polyglot', 'lang_cld3', 'lang_fasttext']]
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(df_test)

NameError: name 'df' is not defined

In [28]:
display(df_test[df_test['lang_fasttext'] != df_test['Language']])

,cleaned,Language,lang,lang_langid,lang_polyglot,lang_cld3,lang_fasttext
8,Service rapide Makla bnina Les sauces chwiaa ....,en,en,fr,en,en,fr
46,so gud food,en,so,en,en,af,de
432,Très beau park de quoi un moment de détente en...,en,fr,fr,fr,fr,fr
648,"Excellent, petit aéroport fonctionnant mieux q...",en,fr,fr,fr,fr,fr
670,V.gooooooooooooooood,en,so,en,en,so,de
707,Good expérience,en,fr,fr,fr,hu,fr
764,Very fast mail,en,en,en,en,en,de


## english error analysis

the **lid.176** model showed the best results on english with an almost perfect predictions. further error analysis showed that the model made mistakes:

- wrong truth labels
- syntax errors 
- short sentence

**note**: from now on i'll only focus on the **lid.176** since we established it is the better classifier

## french error analysis

In [30]:
df_test = df[df['Language'] == 'fr'][['cleaned', 'Language', 'lang', 'lang_langid', 'lang_polyglot', 'lang_cld3', 'lang_fasttext']]
display(df_test[df_test['lang_fasttext'] != df_test['Language']])

,cleaned,Language,lang,lang_langid,lang_polyglot,lang_cld3,lang_fasttext
66,Service impeccable,fr,it,es,en,it,en
68,Professionelle,fr,it,it,fy,it,de
77,Trés agreable,fr,ca,fr,en,fr,en
85,Nuuuuuuuuul 000000000,fr,so,et,en,et,en
128,Service 000,fr,en,en,en,zh,en
139,Service lent,fr,fr,fr,en,fr,en
186,Restaurant sympa,fr,fr,fr,fr,la,en
194,"Delicious Mexican food Nice staff, and calm li...",fr,en,en,en,en,en
207,Bon service,fr,en,en,en,ht,en
231,Services descargots ...,fr,ca,fr,en,fr,en


In [ ]:
the **lid.176** model showed the best results on english with an almost perfect predictions. further error analysis showed that the model made mistakes:

- wrong truth labels
- syntax errors 
- short sentence

In [17]:
text = df_test.iloc[0]['cleaned']
text

'Not salty enough'

In [18]:
import fasttext

model = fasttext.load_model("lid.176.bin")
text = "عندكم عباد في لاكاس يالطيف تشناف ضحكة مكانش"

# Get top 2 language predictions
predictions = model.predict(text, k=3)  # k=2 returns top 2 languages
print(predictions)  

(('__label__ar', '__label__arz', '__label__fa'), array([9.97979522e-01, 1.85876014e-03, 1.08392553e-04]))


In [19]:
%pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [20]:
from langdetect import detect
import nltk
from nltk.tokenize import sent_tokenize

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /home/khaled/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [21]:

text = "C'est une bonne place ولكن الأكل غالي شوية"
sentences = sent_tokenize(text)

for sentence in sentences:
    print(f"Text: {sentence} --> Detected Language: {detect(sentence)}")

Text: C'est une bonne place ولكن الأكل غالي شوية --> Detected Language: fr


In [22]:
sentences

["C'est une bonne place ولكن الأكل غالي شوية"]

In [23]:
import re

text = "C'est une bonne place ولكن الأكل غالي شوية"

# Regex to match Arabic and Latin script separately
arabic_parts = re.findall(r'[\u0600-\u06FF\u0750-\u077F]+', text)
latin_parts = re.findall(r'[a-zA-ZÀ-ÿ]+', text)

print("Arabic:", arabic_parts)
print("Latin:", latin_parts)

Arabic: ['ولكن', 'الأكل', 'غالي', 'شوية']
Latin: ['C', 'est', 'une', 'bonne', 'place']


In [24]:
for part in arabic_parts:
    print(f"Text: {part} --> Detected Language: {detect(part)}")

for part in latin_parts:
    print(f"Text: {part} --> Detected Language: {detect(part)}")

Text: ولكن --> Detected Language: ar
Text: الأكل --> Detected Language: ar
Text: غالي --> Detected Language: ar
Text: شوية --> Detected Language: ar
Text: C --> Detected Language: en
Text: est --> Detected Language: fr
Text: une --> Detected Language: fr
Text: bonne --> Detected Language: cy
Text: place --> Detected Language: es
